<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/04_TADP_Experiment_C_CIFAR10_Cross_Domain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow scikit-learn matplotlib pandas numpy psutil

In [3]:
# ======================================================================================
# TADP EXPERIMENT C — CIFAR-10 CROSS-DOMAIN FEASIBILITY
# FINAL A1/A2/B-ALIGNED DESIGN | K=10 | 2 SEEDS | 4 FL ROUNDS
# ======================================================================================
# Design guarantees:
#   1) Official CIFAR-10 TEST remains untouched until final evaluation.
#   2) Only the official TRAIN split is divided into federated TRAIN/validation data.
#   3) Governance/client selection operates on raw TRAIN client shards BEFORE
#      model normalization or class-weight estimation.
#   4) After participation is fixed, each scenario derives its own TRAIN-only
#      channel mean/std and class weights from its participating TRAIN cohort.
#      Power-of-Choice uses the full TRAIN candidate pool because participation
#      changes round by round.
#   5) The final TADP policy is:
#        all 28 factors present, finite, and >0
#        -> every dimension >=2.5
#        -> HPS <3.0 reject; HPS >=3.5 direct accept
#        -> 3.0<=HPS<3.5: critical-dimension HPS_review over D1,D2,D5,D6
#           using renormalized original HPS weights; accept if HPS_review>=3.25.
#   6) Runtime is measured. Communication is logical application payload.
#      Energy/CO2/cost are reference-scenario estimates, not direct hardware,
#      network, or Google Colab measurements.
#   7) CIFAR-10 is a two-seed descriptive cross-domain feasibility experiment;
#      no statistical-superiority claim is made.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# FINAL TADP POLICY CONSTANTS — A1/A2/B ALIGNED
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
DIMENSION_MIN_FLOOR = 2.5
HPS_REVIEW_THRESHOLD = 3.25
EVIDENCE_VALIDITY_MIN_EXCLUSIVE = 0.0

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

CRITICAL_REVIEW_DIMS = ("dim1", "dim2", "dim5", "dim6")
_CRITICAL_WEIGHT_TOTAL = sum(
    WEIGHTS_PSCORE_DEFAULT[d] for d in CRITICAL_REVIEW_DIMS
)
CRITICAL_REVIEW_WEIGHTS = {
    d: float(WEIGHTS_PSCORE_DEFAULT[d] / _CRITICAL_WEIGHT_TOTAL)
    for d in CRITICAL_REVIEW_DIMS
}
_EXPECTED_CRITICAL_REVIEW_WEIGHTS = {
    "dim1": 0.3125,
    "dim2": 0.1875,
    "dim5": 0.375,
    "dim6": 0.125,
}
assert all(
    math.isclose(
        CRITICAL_REVIEW_WEIGHTS[d],
        expected,
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    for d, expected in _EXPECTED_CRITICAL_REVIEW_WEIGHTS.items()
), f"Unexpected critical review weights: {CRITICAL_REVIEW_WEIGHTS}"

# Human reviewers verify documentary evidence against the predefined rubrics.
# They do not assign HPS/HPS_review and do not make the admission decision.
# No factor-specific hard-veto minima or factor-level review average is used.

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# Factor-count notes:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# two reviewer-driven factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — FINAL A1/A2/B-ALIGNED POLICY
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Controlled documentary evidence for the final A1/A2 policy.

    The factor_minima argument is retained ONLY to report the legacy/reference
    rubric adequacy descriptors in the evidence audit. It is NOT an admission gate.

    For K=10 the frozen bundle library exercises:
      - 4 direct-accept profiles
      - 2 HPS_review-accept profiles
      - 1 evidence-validity reject (a documentary factor = 0)
      - 1 HPS_review reject
      - 1 low-HPS reject
      - 1 dimension-floor reject

    DQ (dim2) is never synthesized here; it remains machine-measured from TRAIN.
    """
    domain = str(domain).lower()
    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    def fill(values_by_dim):
        out = {dim: {} for dim in DOCUMENTARY_DIMS}
        for dim in DOCUMENTARY_DIMS:
            vals = values_by_dim[dim]
            if np.isscalar(vals):
                vals = [float(vals)] * len(FACTOR_NAMES[dim])
            if len(vals) != len(FACTOR_NAMES[dim]):
                raise RuntimeError(f"Bad controlled-evidence template for {dim}.")
            for factor, value in zip(FACTOR_NAMES[dim], vals):
                out[dim][factor] = float(value)
        return out

    def make_profile(role: str, variant: int):
        if role == "DIRECT_STRONG":
            p = fill({
                "dim1": 4.0, "dim3": 4.0, "dim4": 4.0,
                "dim5": 4.0, "dim6": 4.0,
            })
            # Small deterministic variation without changing the intended branch.
            if variant % 2:
                p["dim3"]["data_dictionary"] = 5.0
            if variant % 3 == 2:
                p["dim6"]["ethical_reviews"] = 5.0
            return p

        if role == "HPS_REVIEW_ACCEPT":
            # Critical dimensions are strong enough for HPS_review >=3.25,
            # while D3/D4 are deliberately moderate so the overall HPS remains
            # in the intermediate band for typical CIFAR DQ values.
            return fill({
                "dim1": [3.0, 4.0, 3.0, 4.0],      # mean 3.5
                "dim3": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
                "dim4": [2.0, 3.0, 3.0],           # mean 2.667
                "dim5": [3.0, 3.0, 3.0, 3.0, 3.0], # mean 3.0
                "dim6": [3.0, 4.0, 3.0, 4.0],      # mean 3.5
            })

        if role == "HPS_REVIEW_REJECT":
            # Keep the overall HPS in/near the intermediate band using high
            # non-critical D3/D4, but critical dimensions remain weak enough
            # that HPS_review falls below 3.25.
            return fill({
                "dim1": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
                "dim3": 5.0,
                "dim4": 5.0,
                "dim5": [2.0, 3.0, 2.0, 3.0, 2.5], # mean 2.5
                "dim6": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
            })

        if role == "EVIDENCE_VALIDITY_FAIL":
            p = fill({
                "dim1": 4.0, "dim3": 4.0, "dim4": 4.0,
                "dim5": 4.0, "dim6": 4.0,
            })
            # Final policy Gate 1: all 28 factors must be >0.
            p["dim3"]["version_logs"] = 0.0
            return p

        if role == "LOW_HPS_WEAK":
            return fill({
                "dim1": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
                "dim3": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
                "dim4": [2.0, 3.0, 3.0],           # mean 2.667
                "dim5": [2.0, 3.0, 2.0, 3.0, 2.5], # mean 2.5
                "dim6": [2.0, 3.0, 2.0, 3.0],      # mean 2.5
            })

        if role == "DIMENSION_FLOOR_WEAK":
            p = fill({
                "dim1": 3.5, "dim3": 3.5, "dim4": 2.0,
                "dim5": 3.5, "dim6": 3.5,
            })
            return p

        raise ValueError(f"Unknown evidence role: {role}")

    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("HPS_REVIEW_ACCEPT", 0),
            ("HPS_REVIEW_ACCEPT", 1),
            ("EVIDENCE_VALIDITY_FAIL", 0),
            ("HPS_REVIEW_REJECT", 0),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        archetypes = [
            "DIRECT_STRONG",
            "HPS_REVIEW_ACCEPT",
            "EVIDENCE_VALIDITY_FAIL",
            "HPS_REVIEW_REJECT",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": make_profile(role, int(variant)),
        })

    assignment = rng.permutation(n)
    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]
        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim]) for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                reference_min = float(factor_minima[dim][factor])
                rows.append({
                    "client": str(cid),
                    "bundle_id": bundle["bundle_id"],
                    "evidence_profile": bundle["profile"],
                    "scenario_role": bundle["scenario_role"],
                    "profile_variant": int(bundle["profile_variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5": float(score),
                    "rubric_score_0_5": float(score),
                    "server_mapped_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "human_role": "VERIFY_EVIDENCE_ONLY",
                    "score_assignment": "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by": "SERVER_POLICY",
                    "reference_rubric_adequacy_min_rank": reference_min,
                    "meets_reference_rubric_adequacy": bool(float(score) >= reference_min),
                    "evidence_validity_required": True,
                    "evidence_validity_min_exclusive": float(EVIDENCE_VALIDITY_MIN_EXCLUSIVE),
                    "critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                    "hps_review_threshold_0_5": float(HPS_REVIEW_THRESHOLD),
                    "evidence_artifact_id": f"{cid}-{bundle['bundle_id']}-{dim}-{factor}",
                    "validation_status": "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed": int(evidence_seed),
                    "domain": domain,
                })

    return evidence_by_client, pd.DataFrame(rows)


def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """Mean factor score within each dimension; invalid/missing -> NaN."""
    out = {}
    for dim, expected in FACTOR_NAMES.items():
        vals = []
        valid = True
        for factor in expected:
            if factor not in factors.get(dim, {}):
                valid = False
                break
            try:
                v = float(factors[dim][factor])
            except Exception:
                valid = False
                break
            if not np.isfinite(v):
                valid = False
                break
            vals.append(v)
        out[dim] = float(np.mean(vals)) if valid and vals else np.nan
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    vals = [float(dimensions[d]) for d in weights]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(sum(float(weights[d]) * float(dimensions[d]) for d in weights))


def evidence_validity_audit(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    """All 28 factor scores must be present, finite, and strictly >0."""
    missing, nonfinite, nonpositive = [], [], []
    valid_scores = {}
    for dim, expected in FACTOR_NAMES.items():
        for factor in expected:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue
            try:
                v = float(factors[dim][factor])
            except Exception:
                nonfinite.append(key)
                continue
            if not np.isfinite(v):
                nonfinite.append(key)
                continue
            valid_scores[key] = v
            if v <= EVIDENCE_VALIDITY_MIN_EXCLUSIVE:
                nonpositive.append(f"{key}:{v:.3f}")

    expected_n = int(sum(len(v) for v in FACTOR_NAMES.values()))
    passed = (
        len(valid_scores) == expected_n
        and not missing and not nonfinite and not nonpositive
    )
    return {
        "passed": bool(passed),
        "factor_count_expected": expected_n,
        "factor_count_observed_valid_numeric": int(len(valid_scores)),
        "missing": missing,
        "nonfinite": nonfinite,
        "nonpositive": nonpositive,
    }


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def compute_hps_review(dimensions: Dict[str, float]) -> float:
    vals = [float(dimensions[d]) for d in CRITICAL_REVIEW_DIMS]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(sum(
        CRITICAL_REVIEW_WEIGHTS[d] * float(dimensions[d])
        for d in CRITICAL_REVIEW_DIMS
    ))


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str = "cifar10",
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    Final A1/A2/B-aligned TADP admission policy.

    1) all 28 factors present, finite, and >0
    2) every dimension >=2.5
    3) HPS <3.0 reject; HPS >=3.5 direct accept
    4) intermediate HPS -> HPS_review(D1,D2,D5,D6) >=3.25 accept
    """
    validity = evidence_validity_audit(factors)
    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)
    hps_review = compute_hps_review(dims)
    floor_fail = dimension_floor_failures(dims)

    base = {
        "hps": float(hps) if np.isfinite(hps) else np.nan,
        "hps_review": float(hps_review) if np.isfinite(hps_review) else np.nan,
        "hps_review_threshold_0_5": float(HPS_REVIEW_THRESHOLD),
        "hps_review_margin": (
            float(hps_review - HPS_REVIEW_THRESHOLD)
            if np.isfinite(hps_review) else np.nan
        ),
        "critical_review_dimensions": ";".join(CRITICAL_REVIEW_DIMS),
        "critical_review_weights_json": json.dumps(
            CRITICAL_REVIEW_WEIGHTS, sort_keys=True
        ),
        "dimensions": dims,
        "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures": ";".join(floor_fail),
        "evidence_validity_pass": bool(validity["passed"]),
        "evidence_factor_count_expected": int(validity["factor_count_expected"]),
        "evidence_factor_count_observed_valid_numeric": int(
            validity["factor_count_observed_valid_numeric"]
        ),
        "evidence_missing": ";".join(validity["missing"]),
        "evidence_nonfinite": ";".join(validity["nonfinite"]),
        "evidence_nonpositive": ";".join(validity["nonpositive"]),
    }

    if not validity["passed"]:
        failures = validity["missing"] + validity["nonfinite"] + validity["nonpositive"]
        return {
            **base,
            "decision_path": "EVIDENCE_VALIDITY_GATE",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_EVIDENCE_VALIDITY",
            "reason": "Evidence-validity requirement failed: " + ";".join(failures),
        }

    if floor_fail:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one dimension is below {DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(f"{d}={dims[d]:.3f}" for d in floor_fail)
            ),
        }

    if hps < float(good_cut):
        return {
            **base,
            "decision_path": "LOW_HPS",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
            "reason": f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    if hps >= float(high_cut):
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "initial_action": "AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
            "reason": (
                f"Evidence-validity and dimension-floor checks passed; "
                f"HPS {hps:.3f} >= {high_cut:.3f}"
            ),
        }

    if hps_review >= float(HPS_REVIEW_THRESHOLD):
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "initial_action": "AUTOMATED_REVIEW",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_HPS_REVIEW",
            "reason": (
                f"HPS {hps:.3f} in [{good_cut:.3f},{high_cut:.3f}); "
                f"HPS_review {hps_review:.3f} >= {HPS_REVIEW_THRESHOLD:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "initial_action": "AUTOMATED_REVIEW",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_HPS_REVIEW",
        "reason": (
            f"HPS {hps:.3f} in [{good_cut:.3f},{high_cut:.3f}); "
            f"HPS_review {hps_review:.3f} < {HPS_REVIEW_THRESHOLD:.3f}"
        ),
    }


def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    rows = []
    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }
        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }
        d = tadp_decision(factors, domain=domain)
        row = {
            "run": int(run),
            "domain": str(domain).lower(),
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]) if np.isfinite(d["hps"]) else np.nan,
            "hps_review": float(d["hps_review"]) if np.isfinite(d["hps_review"]) else np.nan,
            "hps_review_threshold_0_5": float(d["hps_review_threshold_0_5"]),
            "hps_review_margin": float(d["hps_review_margin"]) if np.isfinite(d["hps_review_margin"]) else np.nan,
            "critical_review_dimensions": d["critical_review_dimensions"],
            "critical_review_weights_json": d["critical_review_weights_json"],
            "evidence_validity_pass": bool(d["evidence_validity_pass"]),
            "evidence_factor_count_expected": int(d["evidence_factor_count_expected"]),
            "evidence_factor_count_observed_valid_numeric": int(
                d["evidence_factor_count_observed_valid_numeric"]
            ),
            "evidence_missing": d["evidence_missing"],
            "evidence_nonfinite": d["evidence_nonfinite"],
            "evidence_nonpositive": d["evidence_nonpositive"],
            "dimension_min_floor": float(d["dimension_min_floor"]),
            "dimension_floor_failures": d["dimension_floor_failures"],
            "decision_path": d["decision_path"],
            "initial_action": d["initial_action"],
            "final_action": d["final_action"],
            "status": d["status"],
            "reason": d["reason"],
        }
        for dim, value in d["dimensions"].items():
            row[f"{dim}_score_0_5"] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(governance_df: pd.DataFrame) -> List[str]:
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()
    if eligible.empty:
        raise RuntimeError("TADP-SDA cannot select a client: no eligible client.")
    eligible["direct_accept_priority"] = (
        eligible["status"].eq("DIRECT_AUTO_ACCEPTED").astype(int)
    )
    eligible["hps_review_sort"] = pd.to_numeric(
        eligible["hps_review"], errors="coerce"
    ).fillna(-np.inf)
    eligible = eligible.sort_values(
        ["hps", "direct_accept_priority", "hps_review_sort", "client"],
        ascending=[False, False, False, True],
    )
    return [str(eligible.iloc[0]["client"])]


def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """Full 28-factor evidence audit under the final A1/A2 policy."""
    factor_minima = (
        FACTOR_ADEQUACY_MIN_CIFAR10
        if str(domain).lower() == "cifar10"
        else FACTOR_ADEQUACY_MIN_HEALTHCARE
    )
    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index("client", drop=False)
    rows = []

    for cid in map(str, client_ids):
        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(documentary_evidence[cid][dim][factor])
                ref_min = float(factor_minima[dim][factor])
                rows.append({
                    "client": cid,
                    "domain": str(domain).lower(),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value": np.nan,
                    "rubric_score_0_5": score,
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "reference_rubric_adequacy_min_rank": ref_min,
                    "meets_reference_rubric_adequacy": bool(score >= ref_min),
                    "evidence_validity_required": True,
                    "evidence_validity_min_exclusive": float(EVIDENCE_VALIDITY_MIN_EXCLUSIVE),
                    "critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                    "evidence_seed": int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(dq_scores[cid][factor])
            ref_min = float(factor_minima["dim2"][factor])
            raw_col = DQ_RAW_METRIC_BY_FACTOR[factor]
            raw_value = (
                float(dq_index.loc[cid, raw_col])
                if raw_col in dq_index.columns else np.nan
            )
            rows.append({
                "client": cid,
                "domain": str(domain).lower(),
                "dimension": "dim2",
                "dimension_name": DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source": "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value": raw_value,
                "rubric_score_0_5": score,
                "rubric_descriptor": rubric_descriptor("dim2", factor, score),
                "reference_rubric_adequacy_min_rank": ref_min,
                "meets_reference_rubric_adequacy": bool(score >= ref_min),
                "evidence_validity_required": True,
                "evidence_validity_min_exclusive": float(EVIDENCE_VALIDITY_MIN_EXCLUSIVE),
                "critical_review_dimension": True,
                "evidence_seed": int(evidence_seed),
            })

    out = pd.DataFrame(rows)
    expected_rows = len(client_ids) * 28
    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain {expected_rows} rows; found {len(out)}."
        )
    return out


# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) NATIVE TRAIN-ONLY VALIDATOR# GREAT EXPECTATIONS (GX CORE) NATIVE TRAIN-ONLY VALIDATOR
# ======================================================================================
# IMPORTANT:
#   Great Expectations is a DATA-VALIDATION framework, not a client-selection algorithm.
#   This revised cross-domain probe uses native severity-aware TRAIN-only GX validation.
#   GX does not rank contributors and is not forced to match TADP-VR K.
#
#   The GX validation layer covers:
#       - schema
#       - volume
#       - missingness
#       - uniqueness / duplicates
#       - cardinality / category coverage
#       - value / type consistency
#       - distribution
#       - integrity / constraints
#
#   No TEST data, model metric, HPS, HPS_review, or TADP decision is used by GX.
GX_CORE_VERSION = "1.23.0"
GX_NONNULL_TOLERANCE = 0.05
GX_CARDINALITY_COVERAGE = 0.60
GX_DISTRIBUTION_SD_TOLERANCE = 0.75
GX_VALUE_MOSTLY = 0.99


def ensure_great_expectations():
    """
    Import GX Core. In Colab, install the pinned version once if unavailable.
    """
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                f"great_expectations=={GX_CORE_VERSION}",
            ]
        )
        import great_expectations as gx
        return gx


def _gx_meta(category: str, severity: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "policy_severity": str(severity),
        "check_name": str(name),
        "comparator": "GX_CORE_MATCHED_K",
    }


def _gx_add(
    suite,
    specs: List[Dict[str, Any]],
    expectation,
    category: str,
    severity: str,
    name: str,
):
    suite.add_expectation(expectation)
    specs.append({
        "category": str(category),
        "severity": str(severity),
        "name": str(name),
    })


def _aggregate_numeric_reference(
    client_frames: Dict[str, pd.DataFrame],
    numeric_cols: List[str],
) -> Dict[str, Dict[str, float]]:
    """
    TRAIN-only global moments from client-local count/sum/sum-of-squares.
    No TEST data are used.
    """
    out = {}
    for c in numeric_cols:
        count = 0
        total = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += int(len(a))
            total += float(a.sum()) if len(a) else 0.0
            sumsq += float(np.square(a).sum()) if len(a) else 0.0

        if count == 0:
            continue

        mean = total / count
        var = max(0.0, sumsq / count - mean * mean)
        out[c] = {
            "count": float(count),
            "mean": float(mean),
            "std": float(math.sqrt(var)),
        }
    return out


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    """
    TRAIN-only non-null proportions from client-local counts.
    """
    out = {}
    total_rows = float(sum(len(df) for df in client_frames.values()))

    for c in columns:
        nonnull = 0
        for df in client_frames.values():
            if c in df.columns:
                nonnull += int(df[c].notna().sum())
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def _aggregate_category_reference(
    client_frames: Dict[str, pd.DataFrame],
    categorical_cols: List[str],
) -> Dict[str, List[str]]:
    """
    TRAIN-only category support as a union of client-local category sets.
    """
    out = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            if c in df.columns:
                values.update(
                    df[c]
                    .astype("string")
                    .fillna("__MISSING__")
                    .astype(str)
                    .unique()
                    .tolist()
                )
        out[c] = sorted(values)
    return out


def build_gx_healthcare_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: Any,
) -> Dict[str, Any]:
    """
    Build a TRAIN-only reference for GX expectations using federated/local summaries.
    """
    first = next(iter(client_frames.values()))
    original_columns = list(first.columns)

    numeric_cols = list(preprocessor.numeric_cols)
    categorical_cols = list(preprocessor.categorical_cols)

    return {
        "original_columns": original_columns,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_reference": _aggregate_numeric_reference(
            client_frames,
            numeric_cols,
        ),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames,
            list(preprocessor.feature_cols),
        ),
        "category_reference": _aggregate_category_reference(
            client_frames,
            categorical_cols,
        ),
    }


def build_gx_healthcare_validation_frame(
    df: pd.DataFrame,
    preprocessor: Any,
) -> pd.DataFrame:
    """
    Create a GX validation view of a raw client TRAIN partition.

    Original columns remain present. Numeric candidate columns are converted to
    numeric for GX range/distribution expectations, while explicit diagnostic
    columns preserve type-conversion consistency information.
    """
    out = df.copy()

    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()

        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)

    return out


def build_gx_healthcare_suite(
    gx,
    reference: Dict[str, Any],
):
    """
    Build a real GX Expectation Suite for structured healthcare TRAIN data.

    Thresholds are fixed comparator settings, not learned from TEST/model outcomes:
      - non-null proportion may be up to 5 percentage points below the TRAIN reference;
      - categorical support must cover >=60% of TRAIN-wide support;
      - numeric means may deviate by +/-0.75 TRAIN-wide standard deviations.
    """
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_core_suite")
    specs = []

    original_columns = reference["original_columns"]

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=original_columns,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "critical", "required_column_set"),
        ),
        "Schema",
        "critical",
        "required_column_set",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Volume", "warning", "minimum_client_rows"),
        ),
        "Volume",
        "warning",
        "minimum_client_rows",
    )

    if "encounter_id" in original_columns:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Uniqueness", "critical", "encounter_id_unique"),
            ),
            "Uniqueness",
            "critical",
            "encounter_id_unique",
        )

    if "_target" in original_columns:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target",
                value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Integrity", "critical", "target_domain"),
            ),
            "Integrity",
            "critical",
            "target_domain",
        )

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnUniqueValueCountToBeBetween(
                column="_target",
                min_value=3,
                max_value=3,
                severity="warning",
                meta=_gx_meta("Cardinality", "warning", "target_class_cardinality"),
            ),
            "Cardinality",
            "warning",
            "target_class_cardinality",
        )

    # Missingness expectations: compare each feature to TRAIN-only reference.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in original_columns:
            continue
        # Avoid making extremely sparse optional fields dominate the comparator.
        if float(global_nonnull) < 0.10:
            continue

        minimum = max(
            0.0,
            min(
                1.0,
                float(global_nonnull) - GX_NONNULL_TOLERANCE,
            ),
        )

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c,
                min_value=minimum,
                max_value=1.0,
                severity="warning",
                meta=_gx_meta(
                    "Missingness",
                    "warning",
                    f"{c}_nonnull_reference",
                ),
            ),
            "Missingness",
            "warning",
            f"{c}_nonnull_reference",
        )

    # Type/value consistency and numerical distribution.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta(
                    "Type Consistency",
                    "critical",
                    f"{c}_numeric_parseability",
                ),
            ),
            "Type Consistency",
            "critical",
            f"{c}_numeric_parseability",
        )

        ref = reference["numeric_reference"].get(c)
        if ref is None:
            continue

        mu = float(ref["mean"])
        sigma = float(ref["std"])

        if np.isfinite(mu) and np.isfinite(sigma) and sigma > 1e-12:
            lo = mu - GX_DISTRIBUTION_SD_TOLERANCE * sigma
            hi = mu + GX_DISTRIBUTION_SD_TOLERANCE * sigma

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnMeanToBeBetween(
                    column=c,
                    min_value=float(lo),
                    max_value=float(hi),
                    severity="warning",
                    meta=_gx_meta(
                        "Distribution",
                        "warning",
                        f"{c}_mean_reference_band",
                    ),
                ),
                "Distribution",
                "warning",
                f"{c}_mean_reference_band",
            )

    # Domain integrity / range expectations for count-like clinical fields.
    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in original_columns:
            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c,
                    min_value=0.0,
                    mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta(
                        "Integrity",
                        "critical",
                        f"{c}_nonnegative",
                    ),
                ),
                "Integrity",
                "critical",
                f"{c}_nonnegative",
            )

    # Cardinality and allowed-support consistency for categorical features.
    for c, global_values in reference["category_reference"].items():
        if c not in original_columns or not global_values:
            continue

        g = int(len(global_values))

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c,
                value_set=list(global_values),
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta(
                    "Value Consistency",
                    "critical",
                    f"{c}_allowed_train_support",
                ),
            ),
            "Value Consistency",
            "critical",
            f"{c}_allowed_train_support",
        )

        if g >= 2:
            min_unique = max(
                1,
                int(math.ceil(GX_CARDINALITY_COVERAGE * g)),
            )

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnUniqueValueCountToBeBetween(
                    column=c,
                    min_value=min_unique,
                    max_value=g,
                    severity="warning",
                    meta=_gx_meta(
                        "Cardinality",
                        "warning",
                        f"{c}_support_coverage",
                    ),
                ),
                "Cardinality",
                "warning",
                f"{c}_support_coverage",
            )

    return suite, specs


def build_gx_cifar_metadata(
    X: np.ndarray,
    y: np.ndarray,
) -> pd.DataFrame:
    """
    Structured per-image metadata for GX validation of CIFAR-10.
    GX officially supports validating structured metadata derived from
    unstructured data; raw pixels remain client-local.
    """
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    if X.ndim != 4:
        raise RuntimeError(
            f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}"
        )

    finite = np.isfinite(
        X.astype(np.float32)
    ).reshape(len(X), -1).all(axis=1)

    return pd.DataFrame({
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(
            len(X),
            int(X.dtype == np.uint8),
            dtype=np.int8,
        ),
        "finite": finite.astype(np.int8),
        "pixel_min": X.reshape(len(X), -1).min(axis=1).astype(float),
        "pixel_max": X.reshape(len(X), -1).max(axis=1).astype(float),
        "pixel_mean": X.reshape(len(X), -1).mean(axis=1).astype(float),
        "pixel_std": X.reshape(len(X), -1).std(axis=1).astype(float),
    })


def build_gx_cifar_reference(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]],
) -> Dict[str, Any]:
    """
    TRAIN-only image-metadata reference from client-local summary statistics.
    """
    total_n = 0
    sum_mean = 0.0
    sumsq_mean = 0.0
    sum_std = 0.0
    sumsq_std = 0.0

    for X, y in client_raw.values():
        md = build_gx_cifar_metadata(X, y)

        for col in ["pixel_mean", "pixel_std"]:
            a = md[col].to_numpy(dtype=float)
            if col == "pixel_mean":
                sum_mean += float(a.sum())
                sumsq_mean += float(np.square(a).sum())
            else:
                sum_std += float(a.sum())
                sumsq_std += float(np.square(a).sum())

        total_n += len(md)

    n = max(1, int(total_n))

    mean_mu = sum_mean / n
    mean_var = max(0.0, sumsq_mean / n - mean_mu * mean_mu)

    std_mu = sum_std / n
    std_var = max(0.0, sumsq_std / n - std_mu * std_mu)

    return {
        "pixel_mean": {
            "mean": float(mean_mu),
            "std": float(math.sqrt(mean_var)),
        },
        "pixel_std": {
            "mean": float(std_mu),
            "std": float(math.sqrt(std_var)),
        },
    }


def build_gx_cifar_suite(
    gx,
    reference: Dict[str, Any],
):
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_core_suite")
    specs = []

    columns = [
        "label",
        "height",
        "width",
        "channels",
        "dtype_ok",
        "finite",
        "pixel_min",
        "pixel_max",
        "pixel_mean",
        "pixel_std",
    ]

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns,
            exact_match=True,
            severity="critical",
            meta=_gx_meta("Schema", "critical", "image_metadata_schema"),
        ),
        "Schema",
        "critical",
        "image_metadata_schema",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Volume", "warning", "minimum_client_images"),
        ),
        "Volume",
        "warning",
        "minimum_client_images",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label",
            value_set=list(range(10)),
            severity="critical",
            meta=_gx_meta("Integrity", "critical", "label_domain"),
        ),
        "Integrity",
        "critical",
        "label_domain",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectColumnUniqueValueCountToBeBetween(
            column="label",
            min_value=8,
            max_value=10,
            severity="warning",
            meta=_gx_meta("Cardinality", "warning", "label_support"),
        ),
        "Cardinality",
        "warning",
        "label_support",
    )

    for c, value in [
        ("height", 32),
        ("width", 32),
        ("channels", 3),
        ("dtype_ok", 1),
        ("finite", 1),
    ]:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c,
                value_set=[value],
                severity="critical",
                meta=_gx_meta(
                    "Integrity",
                    "critical",
                    f"{c}_constraint",
                ),
            ),
            "Integrity",
            "critical",
            f"{c}_constraint",
        )

    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c,
                min_value=0.0,
                max_value=255.0,
                severity="critical",
                meta=_gx_meta(
                    "Value Consistency",
                    "critical",
                    f"{c}_valid_range",
                ),
            ),
            "Value Consistency",
            "critical",
            f"{c}_valid_range",
        )

    for c in ["pixel_mean", "pixel_std"]:
        mu = float(reference[c]["mean"])
        sigma = float(reference[c]["std"])

        if sigma > 1e-12:
            lo = mu - GX_DISTRIBUTION_SD_TOLERANCE * sigma
            hi = mu + GX_DISTRIBUTION_SD_TOLERANCE * sigma

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnMeanToBeBetween(
                    column=c,
                    min_value=float(lo),
                    max_value=float(hi),
                    severity="warning",
                    meta=_gx_meta(
                        "Distribution",
                        "warning",
                        f"{c}_mean_reference_band",
                    ),
                ),
                "Distribution",
                "warning",
                f"{c}_mean_reference_band",
            )

    return suite, specs


def _gx_result_unexpected_percent(result_obj) -> float:
    try:
        payload = result_obj.result
        if payload is None:
            return float("nan")
        if hasattr(payload, "get"):
            value = payload.get("unexpected_percent", np.nan)
        else:
            value = np.nan
        return float(value) if value is not None else float("nan")
    except Exception:
        return float("nan")


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)
        round_rows.append({
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
        })
        print(
            f"      ✅ ROUND {r}/{len(selected_per_round)} DONE | "
            f"clients={len(selected)} | optimizer_steps={round_steps} | "
            f"cumulative_steps={total_steps}"
        )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    param_b = model_parameter_bytes(global_model)
    ram = monitor.stop()

    # Logical application-level model payload only:
    # one download + one upload per participating client per round.
    communication_b = 0
    for row in round_rows:
        communication_b += int(row["selected_clients"]) * 2 * param_b

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_bytes": int(communication_b),
        "communication_mb": float(communication_b / 1e6),
        "communication_mib": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_bytes": 0,
        "communication_mb": 0.0,
        "communication_mib": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    reference_power_w: float = 12.0,
    grid_carbon_intensity_kg_per_kwh: float = 0.445,
    electricity_price_usd_per_kwh: float = 0.20,
    communication_price_usd_per_mb: float = 0.005,
) -> Dict[str, Any]:
    runtime_s = float(result["runtime_s"])
    communication_bytes = int(result.get("communication_bytes", 0))
    communication_mb = float(result.get("communication_mb", communication_bytes / 1e6))
    communication_mib = float(
        result.get("communication_mib", communication_bytes / (1024 ** 2))
    )

    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": runtime_s,
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_bytes": communication_bytes,
        "communication_mb": communication_mb,
        "communication_mib": communication_mib,
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": str(initial_hash),
        "reference_power_w": float(reference_power_w),
        "grid_carbon_intensity_kg_per_kwh": float(grid_carbon_intensity_kg_per_kwh),
        "electricity_price_usd_per_kwh": float(electricity_price_usd_per_kwh),
        "communication_price_usd_per_mb": float(communication_price_usd_per_mb),
    }

    # Runtime is measured. The following are reference-scenario derived estimates.
    row["energy_wh_per_w"] = float(runtime_s / 3600.0)
    row["energy_wh"] = float(reference_power_w * row["energy_wh_per_w"])
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(
        row["energy_kwh"] * grid_carbon_intensity_kg_per_kwh
    )
    row["energy_cost_usd"] = float(
        row["energy_kwh"] * electricity_price_usd_per_kwh
    )
    row["communication_cost_usd"] = float(
        communication_mb * communication_price_usd_per_mb
    )
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row




def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """Top-K by the same eight machine-measured TADP DQ factors; TEST is never used."""
    rows=[]
    for cid,scores in dq_scores.items():
        vals=[float(scores[f]) for f in FACTOR_NAMES["dim2"]]
        rows.append({
            "client":str(cid),
            "dq_only_score":float(np.mean(vals)),
            "dq_factor_count":int(len(vals)),
        })
    audit=pd.DataFrame(rows).sort_values(
        ["dq_only_score","client"],ascending=[False,True]
    ).reset_index(drop=True)
    audit["dq_only_rank"]=np.arange(1,len(audit)+1)
    selected=audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"]=audit["client"].isin(selected)
    return selected,audit


def local_training_loss_cifar(model: keras.Model, X: np.ndarray, y: np.ndarray) -> float:
    p=model.predict(X,batch_size=512,verbose=0)
    y=np.asarray(y,dtype=np.int32)
    idx=np.arange(len(y))
    probs=np.clip(p[idx,y],1e-12,1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice_cifar(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    batch_size: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
) -> Dict[str, Any]:
    """
    Power-of-Choice cross-domain comparator.
    Each round samples d=min(N,2K) candidates and chooses the K highest TRAIN-loss
    clients. K and total optimizer steps/round are matched to TADP-VR.
    """
    monitor=RAMMonitor().start()
    start=time.perf_counter()
    global_model=build_model_fn()
    global_model.set_weights([np.array(w,copy=True) for w in initial_weights])
    rng=np.random.default_rng(int(run_seed)+880000)
    round_rows=[]
    total_steps=0
    total_selected=0
    communication_b=0
    all_candidates=list(candidate_clients)

    if not (1 <= int(select_k) <= len(all_candidates)):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1,NUM_ROUNDS_FL+1):
        d=min(len(all_candidates),max(int(select_k),int(candidate_multiplier)*int(select_k)))
        if d==len(all_candidates):
            candidate_pool=list(all_candidates)
        else:
            sizes=np.asarray([len(client_arrays[x][1]) for x in all_candidates],dtype=float)
            probs=sizes/sizes.sum()
            candidate_pool=rng.choice(
                all_candidates,size=d,replace=False,p=probs
            ).tolist()

        losses=[]
        for cid in candidate_pool:
            Xc,yc=client_arrays[cid]
            losses.append((str(cid),local_training_loss_cifar(global_model,Xc,yc)))
        losses.sort(key=lambda x:(-x[1],x[0]))
        selected=[cid for cid,_ in losses[:int(select_k)]]
        step_map=allocate_exact_step_budget(
            selected,client_arrays,int(target_steps_per_round),batch_size,LOCAL_EPOCHS
        )

        global_weights=[np.array(w,copy=True) for w in global_model.get_weights()]
        local_weights=[]
        local_sizes=[]
        for j,cid in enumerate(selected):
            Xc,yc=client_arrays[cid]
            local_model=build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model,Xc,yc,steps=int(step_map[cid]),batch_size=batch_size,
                seed=int(run_seed+1000*r+17*j),class_weights=class_weights,
            )
            local_weights.append([np.array(w,copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg=aggregate_weights(local_weights,local_sizes,equal_weight=False)
        global_model=build_model_fn()
        global_model.set_weights(agg)
        round_steps=int(sum(step_map.values()))
        total_steps+=round_steps
        total_selected+=len(selected)
        param_b=model_parameter_bytes(global_model)
        # PoC logical payload:
        #   candidate downloads + one float64 loss per candidate + selected uploads.
        communication_b += (
            len(candidate_pool) * param_b
            + len(candidate_pool) * 8
            + len(selected) * param_b
        )

        print(
            f"PoC round {r}/{NUM_ROUNDS_FL} | candidates={candidate_pool} | "
            f"selected={selected} | steps={round_steps}"
        )
        round_rows.append({
            "round":int(r),
            "candidate_count":int(len(candidate_pool)),
            "candidate_ids":";".join(candidate_pool),
            "selected_clients":int(len(selected)),
            "selected_ids":";".join(selected),
            "optimizer_steps":int(round_steps),
            "cumulative_optimizer_steps":int(total_steps),
            "local_losses":json.dumps({cid:loss for cid,loss in losses},sort_keys=True),
        })

    runtime=float(time.perf_counter()-start)
    metrics=evaluate_model(global_model,X_test,y_test,10)
    ram=monitor.stop()
    return {
        "model":global_model,
        "metrics":metrics,
        "runtime_s":runtime,
        "total_optimizer_steps":int(total_steps),
        "communication_bytes":int(communication_b),
        "communication_mb":float(communication_b/1e6),
        "communication_mib":float(communication_b/(1024**2)),
        "round_audit":round_rows,
        "participants_mean_per_round":float(total_selected/max(1,NUM_ROUNDS_FL)),
        **ram,
    }


# ======================================================================================
# CIFAR-10 — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

def dirichlet_partition_indices(
    y: np.ndarray,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, np.ndarray]:
    y = np.asarray(y).reshape(-1)
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]
        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        if min(len(b) for b in buckets) >= int(min_client_records):
            return {
                cid: np.asarray(b, dtype=np.int64)
                for cid, b in zip(client_ids, buckets)
            }

    raise RuntimeError("Could not obtain valid CIFAR Dirichlet partition.")


def cifar_train_only_mean_std(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Tuple[np.ndarray, np.ndarray]:
    count = 0
    sum_ = np.zeros(3, dtype=np.float64)
    sumsq = np.zeros(3, dtype=np.float64)

    for X, _ in client_raw.values():
        a = X.astype(np.float64) / 255.0
        count += int(np.prod(a.shape[:3]))
        sum_ += a.sum(axis=(0, 1, 2))
        sumsq += np.square(a).sum(axis=(0, 1, 2))

    mean = sum_ / max(1, count)
    var = np.maximum(1e-12, sumsq / max(1, count) - np.square(mean))
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


def cifar_reference_histograms(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, Any]:
    hist = np.zeros((3, 16), dtype=np.float64)
    class_counts = np.zeros(10, dtype=np.float64)

    for X, y in client_raw.values():
        x = X.astype(np.float32) / 255.0
        for ch in range(3):
            h, _ = np.histogram(x[..., ch], bins=np.linspace(0, 1, 17))
            hist[ch] += h
        class_counts += np.bincount(
            np.asarray(y).reshape(-1), minlength=10
        )

    return {"hist": hist, "class_counts": class_counts}


def cifar_dq_scores(
    X: np.ndarray,
    y: np.ndarray,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for CIFAR-10.
    """
    x = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    # 1) Completeness.
    missing_fraction = float(
        np.mean(~np.isfinite(x.astype(np.float32)))
    )
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate — deterministic sample hash audit.
    sample_n = min(len(x), 2000)
    hashes = [
        hashlib.sha1(
            np.ascontiguousarray(x[i]).tobytes()
        ).digest()
        for i in range(sample_n)
    ]
    duplicate_fraction = float(
        1.0 - len(set(hashes)) / max(1, len(hashes))
    )
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    xf = x.astype(np.float32)
    invalid_value = (
        (~np.isfinite(xf))
        | (xf < 0)
        | (xf > 255)
    )
    error_fraction = float(np.mean(invalid_value))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_inconsistency = 0.0 if x.dtype == np.uint8 else 1.0
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        np.mean((y < 0) | (y > 9))
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) RGB distribution consistency against TRAIN-only aggregate.
    xn = xf / 255.0
    jsds = []

    for ch in range(3):
        h, _ = np.histogram(
            xn[..., ch],
            bins=np.linspace(0, 1, 17),
        )
        jsds.append(
            js_divergence(
                h,
                reference["hist"][ch],
            )
        )

    max_jsd = float(max(jsds))
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Class/category coverage.
    present = int(
        np.sum(
            np.bincount(
                y,
                minlength=10,
            ) > 0
        )
    )
    mean_coverage = float(present / 10.0)
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # CIFAR-specific structural expectations:
    #   rank-4 batch, 32x32x3 samples, uint8 raw pixels,
    #   finite values, valid range, valid labels.
    if x.ndim != 4 or x.shape[1:] != (32, 32, 3):
        structural_violation_fraction = 1.0
    else:
        sample_bad = np.zeros(len(x), dtype=bool)

        # Any nonfinite/out-of-range pixel makes that sample structurally invalid.
        pixel_bad = (
            (~np.isfinite(xf))
            | (xf < 0)
            | (xf > 255)
        )
        sample_bad |= np.any(
            pixel_bad.reshape(len(x), -1),
            axis=1,
        )

        if x.dtype != np.uint8:
            sample_bad[:] = True

        if len(y) != len(x):
            structural_violation_fraction = 1.0
        else:
            sample_bad |= ((y < 0) | (y > 9))
            structural_violation_fraction = float(
                np.mean(sample_bad)
            )

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_cifar10_no_leakage(
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    """
    Prepare only RAW TRAIN client shards plus untouched validation/test data.

    Model normalization and class weights are intentionally NOT fitted here.
    They are fitted later, after each scenario's participating cohort is fixed.
    DQ reference statistics are TRAIN-only and may use the full submitted TRAIN
    candidate pool because they belong to pre-training governance, not model
    preprocessing.
    """
    (x_train_official, y_train_official), (x_test, y_test) = \
        keras.datasets.cifar10.load_data()

    y_train_official = y_train_official.reshape(-1).astype(np.int32)
    y_test = y_test.reshape(-1).astype(np.int32)

    idx = np.arange(len(x_train_official))
    train_idx, val_idx = train_test_split(
        idx,
        test_size=0.10,
        stratify=y_train_official,
        random_state=int(split_seed),
    )

    x_train = x_train_official[train_idx]
    y_train = y_train_official[train_idx]
    x_val = x_train_official[val_idx]
    y_val = y_train_official[val_idx]

    parts = dirichlet_partition_indices(
        y_train, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(parts.keys())

    client_raw = {
        cid: (x_train[idxs], y_train[idxs])
        for cid, idxs in parts.items()
    }

    # Governance DQ reference: TRAIN-only full candidate pool.
    dq_reference = cifar_reference_histograms(client_raw)

    dq_scores = {}
    dq_rows = []
    for cid in client_ids:
        Xraw, yraw = client_raw[cid]
        scores, raw_metrics = cifar_dq_scores(Xraw, yraw, dq_reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

    global_train_ids = np.array(
        [f"official_train_{i}" for i in train_idx], dtype=object
    )
    client_ids_namespaced = {
        cid: np.array(
            [f"official_train_{i}" for i in train_idx[idxs]], dtype=object
        )
        for cid, idxs in parts.items()
    }
    global_test_ids = np.array(
        [f"official_test_{i}" for i in range(len(x_test))], dtype=object
    )

    return {
        "client_raw": client_raw,
        "client_ids": client_ids,
        "x_val_raw": x_val,
        "y_val": y_val.astype(np.int32),
        "x_test_raw": x_test,
        "y_test": y_test.astype(np.int32),
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "meta": {
            "official_train_rows": int(len(x_train_official)),
            "federated_train_rows": int(len(x_train)),
            "validation_rows": int(len(x_val)),
            "official_test_rows": int(len(x_test)),
            "n_clients": int(n_clients),
        },
        "client_train_ids": client_ids_namespaced,
        "global_train_ids": global_train_ids,
        "global_test_ids": global_test_ids,
    }


def cifar_preprocessing_fingerprint(
    participating_clients: List[str],
    mean: np.ndarray,
    std: np.ndarray,
    class_weights: Dict[int, float],
) -> str:
    payload = {
        "participating_clients": list(map(str, participating_clients)),
        "rgb_mean": [float(x) for x in np.asarray(mean).reshape(-1)],
        "rgb_std": [float(x) for x in np.asarray(std).reshape(-1)],
        "class_weights": {
            str(int(k)): float(v) for k, v in sorted(class_weights.items())
        },
    }
    canonical = json.dumps(
        payload, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest()


def build_post_selection_cifar_view(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]],
    x_val_raw: np.ndarray,
    y_val: np.ndarray,
    x_test_raw: np.ndarray,
    y_test: np.ndarray,
    participating_clients: List[str],
    view_name: str,
) -> Dict[str, Any]:
    """
    Final Option-A model preprocessing:
      select cohort first -> fit TRAIN-only mean/std and class weights -> transform.

    The official TEST set is transformed only after the TRAIN-only parameters
    have been frozen and never contributes to their estimation.
    """
    participating_clients = list(map(str, participating_clients))
    if not participating_clients:
        raise RuntimeError(f"{view_name}: no participating clients.")

    selected_raw = {
        cid: client_raw[cid] for cid in participating_clients
    }
    mean, std = cifar_train_only_mean_std(selected_raw)

    client_arrays = {}
    y_parts = []
    for cid in participating_clients:
        Xraw, yraw = client_raw[cid]
        X = (Xraw.astype(np.float32) / 255.0 - mean) / std
        client_arrays[cid] = (
            X.astype(np.float32),
            yraw.astype(np.int32),
        )
        y_parts.append(np.asarray(yraw, dtype=np.int32))

    y_train = np.concatenate(y_parts)
    cw = class_weight_dict(y_train)

    X_val = (x_val_raw.astype(np.float32) / 255.0 - mean) / std
    X_test = (x_test_raw.astype(np.float32) / 255.0 - mean) / std

    fp = cifar_preprocessing_fingerprint(
        participating_clients, mean, std, cw
    )

    return {
        "view_name": str(view_name),
        "participating_clients": participating_clients,
        "client_arrays": client_arrays,
        "X_val": X_val.astype(np.float32),
        "y_val": np.asarray(y_val, dtype=np.int32),
        "X_test": X_test.astype(np.float32),
        "y_test": np.asarray(y_test, dtype=np.int32),
        "class_weights": cw,
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
        "preprocessing_sha256": fp,
        "train_rows": int(len(y_train)),
    }


def build_cifar_model(lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(32, 32, 3), dtype=tf.float32)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(10, activation="softmax", dtype=tf.float32)(x)

    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL CROSS-DOMAIN REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """Summarize the frozen controlled documentary-evidence assignment."""
    required = {
        "client", "bundle_id", "evidence_profile", "scenario_role",
        "profile_variant", "factor", "rubric_score_0_5", "evidence_seed",
    }
    missing = sorted(required - set(evidence_df.columns))
    if missing:
        raise RuntimeError(
            f"Controlled evidence table missing columns: {missing}"
        )

    summary = (
        evidence_df.groupby(["client", "bundle_id"], as_index=False)
        .agg(
            evidence_profile=("evidence_profile", "first"),
            scenario_role=("scenario_role", "first"),
            profile_variant=("profile_variant", "first"),
            documentary_factor_count=("factor", "count"),
            documentary_mean_score=("rubric_score_0_5", "mean"),
            documentary_min_score=("rubric_score_0_5", "min"),
            documentary_max_score=("rubric_score_0_5", "max"),
            evidence_seed=("evidence_seed", "first"),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )
    expected = sum(len(FACTOR_NAMES[d]) for d in DOCUMENTARY_DIMS)
    if not summary["documentary_factor_count"].eq(expected).all():
        raise RuntimeError(
            f"Expected {expected} documentary factors per client."
        )
    return summary


def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    print_banner("FINAL TADP POLICY — EVIDENCE VALIDITY + DIMENSION FLOOR + HPS + HPS_review")
    print(
        "Gate 1: all 28 factors must be present, finite, and >0.\n"
        f"Gate 2: every dimension must be >= {DIMENSION_MIN_FLOOR:.1f}/5.\n"
        f"HPS: <{GOOD_CUT:.1f}=Reject; [{GOOD_CUT:.1f},{HIGH_CUT:.1f})=Automated review; "
        f">={HIGH_CUT:.1f}=Direct accept.\n"
        f"HPS_review uses {CRITICAL_REVIEW_DIMS} with weights "
        f"{CRITICAL_REVIEW_WEIGHTS}; accept if >= {HPS_REVIEW_THRESHOLD:.2f}."
    )
    print(
        "Human role: documentary-evidence verification only; "
        "factor mapping, HPS/HPS_review, and final admission are automated."
    )

    dq_cols = [
        "client", "completeness", "duplication_rate",
        "value_validity_error_rate", "type_consistency", "label_integrity",
        "feature_distribution_consistency", "feature_category_coverage",
        "structural_constraint_integrity",
    ]
    print_banner("TRAIN-ONLY DATA QUALITY — 8 MACHINE-MEASURED FACTORS")
    print(dq[[c for c in dq_cols if c in dq.columns]].to_string(index=False))

    display_cols = [
        "client", "evidence_validity_pass", "evidence_nonpositive",
        "hps", "hps_review", "hps_review_threshold_0_5",
        "dimension_floor_failures",
        "dim1_score_0_5", "dim2_score_0_5", "dim3_score_0_5",
        "dim4_score_0_5", "dim5_score_0_5", "dim6_score_0_5",
        "decision_path", "final_action", "status", "reason",
    ]
    print_banner("FROZEN TADP GOVERNANCE")
    print(gov[[c for c in display_cols if c in gov.columns]].to_string(index=False))
    print(f"\nFrozen TADP-VR cohort: {len(vr_clients)}/{len(gov)} -> {vr_clients}")
    print(f"TADP-SDA audit client: {sda_clients}")

    print_banner("GX CORE TRAIN-ONLY NATIVE VALIDATOR")
    gx_cols = [
        "client", "gx_native_suite_success", "gx_operational_valid",
        "gx_critical_failures", "gx_warning_failures",
        "ge_expectations_passed", "ge_expectations_total",
        "ge_pass_rate", "ge_final_action",
    ]
    print(ge[[c for c in gx_cols if c in ge.columns]].to_string(index=False))
    print(f"GX operationally valid clients: {len(ge_clients)}/{len(ge)} -> {ge_clients}")
    print("GX is a validator, not a matched-K selection method.")


def build_hash_chained_governance_ledger(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    output_path: Path,
):
    """
    Prototype SHA-256 forward hash chain for tamper evidence.
    This is not described as immutable storage or a blockchain.
    """
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in gov.sort_values("client").iterrows():
        seq += 1
        payload = {
            "sequence": seq,
            "governance_system": "TADP",
            "client": str(r["client"]),
            "hps": float(r["hps"]) if pd.notna(r["hps"]) else None,
            "hps_review": float(r["hps_review"]) if pd.notna(r["hps_review"]) else None,
            "evidence_validity_pass": bool(r["evidence_validity_pass"]),
            "evidence_nonpositive": str(r["evidence_nonpositive"]),
            "dimension_floor_failures": str(r["dimension_floor_failures"]),
            "decision_path": str(r["decision_path"]),
            "final_action": str(r["final_action"]),
            "status": str(r["status"]),
            "reason": str(r["reason"]),
            "previous_hash": prev_hash,
        }
        canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
        entry_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        payload["entry_hash"] = entry_hash
        rows.append(payload)
        prev_hash = entry_hash

    for _, r in ge.sort_values("client").iterrows():
        seq += 1
        payload = {
            "sequence": seq,
            "governance_system": "Great Expectations GX Core native validator",
            "client": str(r["client"]),
            "hps": None,
            "hps_review": None,
            "evidence_validity_pass": None,
            "evidence_nonpositive": None,
            "dimension_floor_failures": None,
            "decision_path": "RULE_BASED_VALIDATION",
            "final_action": str(r["ge_final_action"]),
            "status": "GX_NATIVE_TRAIN_ONLY",
            "reason": (
                f"{int(r['ge_expectations_passed'])}/{int(r['ge_expectations_total'])} "
                "GX expectations passed; "
                f"critical_failures={int(r['gx_critical_failures'])}; "
                f"warning_failures={int(r['gx_warning_failures'])}"
            ),
            "previous_hash": prev_hash,
        }
        canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
        entry_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        payload["entry_hash"] = entry_hash
        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_path, index=False)
    return out


def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_bytes": int(result.get("communication_bytes", 0)),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh_per_w", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "communication_mib", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="cifar10",
):
    """Governance-only sensitivity diagnostic under the final admission policy."""
    rows = []
    for j in range(int(n_realizations)):
        seed = int(base_seed + j)
        evidence, _ = generate_controlled_documentary_evidence(
            client_ids,
            seed,
            FACTOR_ADEQUACY_MIN_CIFAR10,
            domain=domain,
        )
        gov = build_tadp_governance(
            client_ids, evidence, dq_scores,
            run=0, evidence_seed=seed, domain=domain,
        )
        accepted = accepted_tadp_vr(gov)
        rows.append({
            "realization": j + 1,
            "evidence_seed": seed,
            "accepted_count": len(accepted),
            "accepted_clients": ";".join(accepted),
            "mean_hps": float(gov["hps"].mean()),
            "mean_hps_review": float(gov["hps_review"].mean()),
            "evidence_validity_rejects": int(
                gov["status"].eq("AUTO_REJECTED_EVIDENCE_VALIDITY").sum()
            ),
            "dimension_floor_rejects": int(
                gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()
            ),
            "low_hps_rejects": int(
                gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()
            ),
            "hps_review_accepts": int(
                gov["status"].eq("ACCEPTED_AFTER_HPS_REVIEW").sum()
            ),
            "hps_review_rejects": int(
                gov["status"].eq("AUTO_REJECTED_HPS_REVIEW").sum()
            ),
            "direct_auto_accepts": int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)


def cifar_partition_table(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> pd.DataFrame:
    rows = []

    for cid, (_, y) in client_arrays.items():
        counts = np.bincount(
            np.asarray(y).reshape(-1),
            minlength=10,
        )

        row = {
            "client": cid,
            "records": int(len(y)),
        }
        for k in range(10):
            row[f"class_{k}"] = int(counts[k])

        rows.append(row)

    return pd.DataFrame(rows)




# ======================================================================================
# EXPERIMENT C — FINAL CROSS-DOMAIN FEASIBILITY CONFIGURATION
# ======================================================================================

EXPERIMENT_VERSION = (
    "TADP-C-FINAL-A1A2B-OPTIONA-CIFAR10-K10-2SEED-4ROUND-"
    "28F-VALIDITY-DIMFLOOR-HPSREVIEW"
)

TRAINING_RUN_SEEDS = [42, 142]
NUM_CLIENTS = 10
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
FEDPROX_MU = 0.01
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 8101
CLIENT_PARTITION_SEED = 8201
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 3042
FROZEN_RANDOMK_SELECTION_SEED = 4042

REFERENCE_POWER_W = 12.0
GRID_CARBON_INTENSITY_KG_PER_KWH = 0.445
ELECTRICITY_PRICE_USD_PER_KWH = 0.20
COMMUNICATION_PRICE_USD_PER_MB = 0.005

USE_GOOGLE_DRIVE_CHECKPOINTS = True

CIFAR_SCENARIOS = [
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP-VR Federated",
]
assert len(CIFAR_SCENARIOS) == 6

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_C_FINAL_CIFAR10_OPTIONA_2SEED_4ROUND",
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


def main():
    print_banner(EXPERIMENT_VERSION)
    print(f"Training seeds: {TRAINING_RUN_SEEDS}")
    print(f"FL rounds: {NUM_ROUNDS_FL}")
    print(f"Clients: {NUM_CLIENTS}")
    print(
        "Final TADP policy: 28-factor evidence validity -> dimension floor -> "
        "HPS bands -> critical-dimension HPS_review."
    )
    print(
        "Cross-domain interpretation: two-seed descriptive feasibility evidence; "
        "no statistical-superiority claim."
    )

    # ------------------------------------------------------------------
    # 1) Raw TRAIN/validation/TEST preparation. TEST remains untouched.
    # ------------------------------------------------------------------
    data = prepare_cifar10_no_leakage(
        split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=NUM_CLIENTS,
        alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT,
        data["global_train_ids"],
        data["global_test_ids"],
        data["client_train_ids"],
        extra={
            "official_test_untouched_before_final_evaluation": True,
            "validation_excluded_from_client_partition": True,
            "governance_before_model_normalization": True,
            "scenario_specific_train_only_normalization": True,
            "scenario_specific_train_only_class_weights": True,
            "dq_gx_tadp_train_only": True,
        },
    )
    print_banner("NO-LEAKAGE AUDIT")
    print(pd.DataFrame([leakage]).to_string(index=False))

    pd.DataFrame([data["meta"]]).to_json(
        EXPERIMENT_ROOT / "dataset_metadata.json",
        orient="records",
        indent=2,
    )

    partition_df = cifar_partition_table(data["client_raw"])
    partition_df.to_csv(
        EXPERIMENT_ROOT / "client_partition_audit.csv", index=False
    )
    data["dq_audit"].to_csv(
        EXPERIMENT_ROOT / "dq_train_only_audit.csv", index=False
    )

    client_ids = list(map(str, data["client_ids"]))
    all_clients = list(client_ids)

    print_banner("FROZEN CIFAR-10 RAW TRAIN CLIENT PARTITIONS")
    print(partition_df.to_string(index=False))

    # ------------------------------------------------------------------
    # 2) Frozen governance evidence + final TADP policy.
    # ------------------------------------------------------------------
    documentary_evidence, evidence_df = generate_controlled_documentary_evidence(
        client_ids,
        FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        FACTOR_ADEQUACY_MIN_CIFAR10,
        domain="cifar10",
    )
    evidence_df.to_csv(
        EXPERIMENT_ROOT / "controlled_documentary_evidence_matrix.csv",
        index=False,
    )
    evidence_summary = evidence_assignment_summary(evidence_df)
    evidence_summary.to_csv(
        EXPERIMENT_ROOT / "evidence_assignment_summary.csv",
        index=False,
    )

    gov = build_tadp_governance(
        client_ids,
        documentary_evidence,
        data["dq_scores"],
        run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="cifar10",
    )
    gov.to_csv(
        EXPERIMENT_ROOT / "tadp_final_governance_frozen.csv",
        index=False,
    )

    full_evidence = build_full_factor_evidence_table(
        client_ids=client_ids,
        documentary_evidence=documentary_evidence,
        dq_scores=data["dq_scores"],
        dq_audit=data["dq_audit"],
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="cifar10",
    )
    full_evidence.to_csv(
        EXPERIMENT_ROOT / "all_28_factor_evidence_by_client.csv",
        index=False,
    )

    policy_factor_rows = []
    for dim in FACTOR_NAMES:
        for factor in FACTOR_NAMES[dim]:
            policy_factor_rows.append({
                "domain": "cifar10",
                "dimension": dim,
                "dimension_name": DIMENSION_NAMES[dim],
                "factor": factor,
                "evidence_validity_required": True,
                "evidence_validity_min_exclusive": float(EVIDENCE_VALIDITY_MIN_EXCLUSIVE),
                "critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                "critical_review_weight": (
                    float(CRITICAL_REVIEW_WEIGHTS[dim])
                    if dim in CRITICAL_REVIEW_WEIGHTS else np.nan
                ),
                "dimension_floor_0_5": float(DIMENSION_MIN_FLOOR),
                "hps_reject_threshold_0_5": float(GOOD_CUT),
                "hps_direct_accept_threshold_0_5": float(HIGH_CUT),
                "hps_review_threshold_0_5": float(HPS_REVIEW_THRESHOLD),
            })
    pd.DataFrame(policy_factor_rows).to_csv(
        EXPERIMENT_ROOT / "cifar10_final_admission_policy.csv",
        index=False,
    )

    vr_clients = accepted_tadp_vr(gov)
    sda_clients = accepted_tadp_sda(gov)

    if not vr_clients:
        raise RuntimeError("CIFAR TADP-VR admitted zero clients. Fail closed.")
    if len(vr_clients) >= len(all_clients):
        raise RuntimeError(
            "CIFAR controlled governance admitted all clients; "
            "the cross-domain selection control would be uninformative."
        )

    status_counts = gov["status"].value_counts().to_dict()
    pd.DataFrame([
        {"status": k, "count": int(v)}
        for k, v in sorted(status_counts.items())
    ]).to_csv(
        EXPERIMENT_ROOT / "governance_branch_coverage.csv", index=False
    )

    # GX remains a separate TRAIN-only technical validator, not a forced-K selector.
    ge, ge_expectation_audit = ge_governance(
        client_data=data["client_raw"],
        client_ids=client_ids,
        accept_count=len(vr_clients),
        domain="cifar10",
        dq_scores=data["dq_scores"],
    )
    ge.to_csv(
        EXPERIMENT_ROOT / "gx_core_train_only_governance_frozen.csv",
        index=False,
    )
    ge_expectation_audit.to_csv(
        EXPERIMENT_ROOT / "gx_core_expectation_audit.csv",
        index=False,
    )
    ge_clients = ge.loc[
        ge["ge_final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()

    # Matched Random-K and DQ-only use the observed TADP-VR K.
    rng = np.random.default_rng(FROZEN_RANDOMK_SELECTION_SEED)
    randomk_clients = rng.choice(
        client_ids, size=len(vr_clients), replace=False
    ).tolist()
    dq_only_clients, dq_only_audit = select_dq_only_clients(
        data["dq_scores"], len(vr_clients)
    )
    dq_only_audit.to_csv(
        EXPERIMENT_ROOT / "dq_only_selection_audit.csv", index=False
    )

    print_governance_details(
        gov, ge, data["dq_audit"],
        vr_clients, sda_clients, ge_clients,
        domain="cifar10",
    )

    build_hash_chained_governance_ledger(
        gov, ge,
        EXPERIMENT_ROOT / "tamper_evident_governance_hash_chain.csv",
    )

    frozen_plan = {
        "experiment_version": EXPERIMENT_VERSION,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed": FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "frozen_randomk_selection_seed": FROZEN_RANDOMK_SELECTION_SEED,
        "all_clients": all_clients,
        "gx_clients": ge_clients,
        "tadp_vr_clients": vr_clients,
        "randomk_clients": randomk_clients,
        "dq_only_clients": dq_only_clients,
        "final_policy": (
            "28-factor validity >0 -> dimension floor 2.5 -> "
            "HPS <3 reject / >=3.5 direct accept -> HPS_review >=3.25"
        ),
        "critical_review_dimensions": list(CRITICAL_REVIEW_DIMS),
        "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
        "option_a_preprocessing": True,
    }
    atomic_write_json(
        frozen_plan, EXPERIMENT_ROOT / "frozen_cohort_plan.json"
    )

    # ------------------------------------------------------------------
    # 3) Scenario-specific post-selection preprocessing.
    # ------------------------------------------------------------------
    scenario_cohorts = {
        "Vanilla FedAvg": all_clients,
        "FedProx": all_clients,
        "Random-K": randomk_clients,
        "DQ-only Federated": dq_only_clients,
        # PoC changes selected participants round by round, so its preprocessing
        # uses the full TRAIN candidate pool, matching the final A1 exception.
        "Power-of-Choice": all_clients,
        "TADP-VR Federated": vr_clients,
    }

    scenario_views = {}
    preproc_rows = []

    for scenario, cohort in scenario_cohorts.items():
        view = build_post_selection_cifar_view(
            client_raw=data["client_raw"],
            x_val_raw=data["x_val_raw"],
            y_val=data["y_val"],
            x_test_raw=data["x_test_raw"],
            y_test=data["y_test"],
            participating_clients=cohort,
            view_name=scenario,
        )
        scenario_views[scenario] = view
        preproc_rows.append({
            "scenario": scenario,
            "participating_clients": ";".join(cohort),
            "participating_client_count": len(cohort),
            "train_rows": int(view["train_rows"]),
            "rgb_mean_r": float(view["mean"][0]),
            "rgb_mean_g": float(view["mean"][1]),
            "rgb_mean_b": float(view["mean"][2]),
            "rgb_std_r": float(view["std"][0]),
            "rgb_std_g": float(view["std"][1]),
            "rgb_std_b": float(view["std"][2]),
            "class_weights_json": json.dumps(view["class_weights"], sort_keys=True),
            "preprocessing_sha256": view["preprocessing_sha256"],
            "preprocessing_fit_scope": (
                "FULL_TRAIN_CANDIDATE_POOL"
                if scenario == "Power-of-Choice"
                else "PARTICIPATING_TRAIN_COHORT"
            ),
        })

    preproc_df = pd.DataFrame(preproc_rows)
    preproc_df.to_csv(
        EXPERIMENT_ROOT / "scenario_preprocessing_audit.csv",
        index=False,
    )
    print_banner("OPTION-A SCENARIO PREPROCESSING AUDIT")
    print(preproc_df.to_string(index=False))

    # Full-participation scenarios must share identical preprocessing.
    full_hashes = {
        scenario_views[x]["preprocessing_sha256"]
        for x in ["Vanilla FedAvg", "FedProx", "Power-of-Choice"]
    }
    if len(full_hashes) != 1:
        raise RuntimeError("Full-pool CIFAR preprocessing fingerprints diverged.")

    # ------------------------------------------------------------------
    # 4) Exact step plans.
    # ------------------------------------------------------------------
    def natural_map(scenario, cohort):
        arrays = scenario_views[scenario]["client_arrays"]
        return {
            c: natural_steps(
                len(arrays[c][1]), BATCH_SIZE, LOCAL_EPOCHS
            )
            for c in cohort
        }

    step_map_all = natural_map("Vanilla FedAvg", all_clients)
    step_map_vr = natural_map("TADP-VR Federated", vr_clients)
    vr_target_per_round = int(sum(step_map_vr.values()))

    step_map_randomk = allocate_exact_step_budget(
        randomk_clients,
        scenario_views["Random-K"]["client_arrays"],
        vr_target_per_round,
        BATCH_SIZE,
        LOCAL_EPOCHS,
    )
    step_map_dq_only = allocate_exact_step_budget(
        dq_only_clients,
        scenario_views["DQ-only Federated"]["client_arrays"],
        vr_target_per_round,
        BATCH_SIZE,
        LOCAL_EPOCHS,
    )

    parity_rows = []
    for r in range(1, NUM_ROUNDS_FL + 1):
        parity_rows.append({
            "round": r,
            "tadp_vr_k": len(vr_clients),
            "randomk_k": len(randomk_clients),
            "dq_only_k": len(dq_only_clients),
            "tadp_vr_steps": vr_target_per_round,
            "randomk_steps": int(sum(step_map_randomk.values())),
            "dq_only_steps": int(sum(step_map_dq_only.values())),
            "randomk_same_k": len(vr_clients) == len(randomk_clients),
            "randomk_exact_step_parity": (
                vr_target_per_round == int(sum(step_map_randomk.values()))
            ),
            "dq_only_same_k": len(vr_clients) == len(dq_only_clients),
            "dq_only_exact_step_parity": (
                vr_target_per_round == int(sum(step_map_dq_only.values()))
            ),
        })
    parity_df = pd.DataFrame(parity_rows)
    parity_df.to_csv(
        EXPERIMENT_ROOT / "matched_selection_parity_plan.csv", index=False
    )
    if not parity_df[
        [
            "randomk_same_k", "randomk_exact_step_parity",
            "dq_only_same_k", "dq_only_exact_step_parity",
        ]
    ].all().all():
        raise RuntimeError("Matched-selection parity plan failed.")

    step_maps = {
        "Vanilla FedAvg": [dict(step_map_all) for _ in range(NUM_ROUNDS_FL)],
        "FedProx": [dict(step_map_all) for _ in range(NUM_ROUNDS_FL)],
        "Random-K": [dict(step_map_randomk) for _ in range(NUM_ROUNDS_FL)],
        "DQ-only Federated": [dict(step_map_dq_only) for _ in range(NUM_ROUNDS_FL)],
        "TADP-VR Federated": [dict(step_map_vr) for _ in range(NUM_ROUNDS_FL)],
    }

    # ------------------------------------------------------------------
    # 5) Two-seed, four-round descriptive cross-domain training.
    # ------------------------------------------------------------------
    completed = set(
        load_checkpoint_state(CHECKPOINT_STATE).get("completed", [])
    )
    selection_rows = []

    build_model_fn = lambda: build_cifar_model(LEARNING_RATE)

    for run_idx, seed in enumerate(TRAINING_RUN_SEEDS, start=1):
        print_banner(
            f"CIFAR RUN {run_idx}/{len(TRAINING_RUN_SEEDS)} | seed={seed}"
        )
        seed_everything(seed)
        init_model = build_model_fn()
        initial_weights = [
            np.array(w, copy=True) for w in init_model.get_weights()
        ]
        initial_hash = sha256_weights(initial_weights)
        del init_model
        tf.keras.backend.clear_session()
        gc.collect()

        for scenario_idx, scenario in enumerate(CIFAR_SCENARIOS, start=1):
            key = (
                f"version={EXPERIMENT_VERSION}|run={run_idx}|seed={seed}|"
                f"scenario={scenario}|pre={scenario_views[scenario]['preprocessing_sha256']}"
            )
            print_banner(
                f"SCENARIO {scenario_idx}/{len(CIFAR_SCENARIOS)} | {scenario}"
            )

            if key in completed:
                print(f"CHECKPOINT FOUND -> skipping {scenario}")
                continue

            view = scenario_views[scenario]
            arrays = view["client_arrays"]
            class_weights = view["class_weights"]
            X_test = view["X_test"]
            y_test = view["y_test"]

            if scenario == "Power-of-Choice":
                selection_rows.append({
                    "run": run_idx,
                    "seed": seed,
                    "scenario": scenario,
                    "selected_clients": "DYNAMIC_PER_ROUND",
                    "selected_count": len(vr_clients),
                    "selection_frozen_across_rounds": False,
                })
                seed_everything(seed)
                result = federated_train_power_of_choice_cifar(
                    build_model_fn=build_model_fn,
                    initial_weights=initial_weights,
                    candidate_clients=all_clients,
                    select_k=len(vr_clients),
                    target_steps_per_round=vr_target_per_round,
                    client_arrays=arrays,
                    X_test=X_test,
                    y_test=y_test,
                    batch_size=BATCH_SIZE,
                    class_weights=class_weights,
                    run_seed=seed,
                )
                selected_for_row = sorted(set(
                    cid
                    for rr in result["round_audit"]
                    for cid in str(rr["selected_ids"]).split(";")
                    if cid
                ))
                selected_count = len(vr_clients)
            else:
                selected = list(scenario_cohorts[scenario])
                selection_rows.append({
                    "run": run_idx,
                    "seed": seed,
                    "scenario": scenario,
                    "selected_clients": ";".join(selected),
                    "selected_count": len(selected),
                    "selection_frozen_across_rounds": True,
                })
                selected_per_round = [
                    list(selected) for _ in range(NUM_ROUNDS_FL)
                ]
                seed_everything(seed)
                result = federated_train(
                    build_model_fn=build_model_fn,
                    initial_weights=initial_weights,
                    selected_per_round=selected_per_round,
                    client_arrays=arrays,
                    X_test=X_test,
                    y_test=y_test,
                    n_classes=10,
                    batch_size=BATCH_SIZE,
                    local_epochs=LOCAL_EPOCHS,
                    class_weights=class_weights,
                    run_seed=seed,
                    exact_step_maps=step_maps[scenario],
                    equal_weight=False,
                    fedprox_mu=(
                        FEDPROX_MU if scenario == "FedProx" else 0.0
                    ),
                )
                selected_for_row = selected
                selected_count = len(selected)

            row = result_row(
                run=run_idx,
                seed=seed,
                scenario=scenario,
                result=result,
                initial_hash=initial_hash,
                reference_power_w=REFERENCE_POWER_W,
                grid_carbon_intensity_kg_per_kwh=GRID_CARBON_INTENSITY_KG_PER_KWH,
                electricity_price_usd_per_kwh=ELECTRICITY_PRICE_USD_PER_KWH,
                communication_price_usd_per_mb=COMMUNICATION_PRICE_USD_PER_MB,
            )
            row.update({
                "experiment_version": EXPERIMENT_VERSION,
                "selected_client_ids": ";".join(selected_for_row),
                "selected_client_count": int(selected_count),
                "fl_rounds": int(NUM_ROUNDS_FL),
                "preprocessing_sha256": view["preprocessing_sha256"],
                "preprocessing_fit_scope": (
                    "FULL_TRAIN_CANDIDATE_POOL"
                    if scenario == "Power-of-Choice"
                    else "PARTICIPATING_TRAIN_COHORT"
                ),
                "rgb_mean_json": json.dumps(view["mean"].tolist()),
                "rgb_std_json": json.dumps(view["std"].tolist()),
                "class_weights_json": json.dumps(view["class_weights"], sort_keys=True),
                "evidence_validity_min_exclusive": float(EVIDENCE_VALIDITY_MIN_EXCLUSIVE),
                "dimension_min_floor_0_5": float(DIMENSION_MIN_FLOOR),
                "hps_review_threshold_0_5": float(HPS_REVIEW_THRESHOLD),
                "frozen_evidence_assignment_seed": int(FROZEN_EVIDENCE_ASSIGNMENT_SEED),
            })

            upsert_csv(
                row, PERF_CHECKPOINT, ["run", "scenario"]
            )
            write_scenario_checkpoint(
                EXPERIMENT_ROOT, run_idx, scenario, result
            )
            mark_checkpoint_complete(
                CHECKPOINT_STATE,
                key,
                extra={
                    "experiment_version": EXPERIMENT_VERSION,
                    "run": run_idx,
                    "seed": seed,
                    "scenario": scenario,
                },
            )
            completed.add(key)

            print(
                f"TEST | Acc={row['accuracy']:.6f} | "
                f"F1={row['f1_macro']:.6f} | "
                f"AUC={row['roc_auc_ovr_macro']:.6f}"
            )
            print(
                f"Runtime={row['runtime_s']:.2f}s | "
                f"Communication={row['communication_mb']:.3f} MB "
                f"({row['communication_mib']:.3f} MiB) | "
                f"Est. energy={row['energy_wh']:.6f} Wh | "
                f"Est. CO2={row['co2_kg']:.9f} kg"
            )

            del result["model"]
            tf.keras.backend.clear_session()
            gc.collect()

    # ------------------------------------------------------------------
    # 6) Final audits and outputs.
    # ------------------------------------------------------------------
    perf = pd.read_csv(PERF_CHECKPOINT)
    expected = len(TRAINING_RUN_SEEDS) * len(CIFAR_SCENARIOS)
    if len(perf[["run", "scenario"]].drop_duplicates()) != expected:
        raise RuntimeError(
            f"Incomplete CIFAR run: expected {expected} run-scenario results."
        )

    perf = perf.sort_values(["run", "scenario"]).reset_index(drop=True)
    perf.to_csv(
        EXPERIMENT_ROOT / "performance_metrics.csv", index=False
    )

    summary = summarize_runs_with_ci(perf)
    summary.to_csv(
        EXPERIMENT_ROOT / "performance_summary_mean_sd_ci95.csv",
        index=False,
    )

    # Within each seed the CNN input shape is fixed, so the same W0 must be used
    # across all six scenarios.
    init_audit_rows = []
    for run_idx, d in perf.groupby("run"):
        n_hash = d["initial_weights_sha256"].nunique()
        init_audit_rows.append({
            "run": int(run_idx),
            "unique_initial_weight_hashes": int(n_hash),
            "initialization_parity_pass": bool(n_hash == 1),
        })
    init_audit = pd.DataFrame(init_audit_rows)
    init_audit.to_csv(
        EXPERIMENT_ROOT / "initialization_parity_audit.csv", index=False
    )
    if not init_audit["initialization_parity_pass"].all():
        raise RuntimeError("CIFAR within-seed initialization parity failed.")

    # Matched selection fairness.
    fairness_rows = []
    for run_idx in sorted(perf["run"].unique()):
        vr = perf[
            perf["run"].eq(run_idx)
            & perf["scenario"].eq("TADP-VR Federated")
        ].iloc[0]
        for baseline in ["Random-K", "DQ-only Federated", "Power-of-Choice"]:
            br = perf[
                perf["run"].eq(run_idx)
                & perf["scenario"].eq(baseline)
            ].iloc[0]
            fairness_rows.append({
                "run": int(run_idx),
                "baseline": baseline,
                "same_k": int(vr["selected_client_count"]) == int(br["selected_client_count"]),
                "exact_total_step_parity": int(vr["optimizer_steps"]) == int(br["optimizer_steps"]),
                "same_initial_weights": str(vr["initial_weights_sha256"]) == str(br["initial_weights_sha256"]),
            })
    fairness = pd.DataFrame(fairness_rows)
    fairness.to_csv(
        EXPERIMENT_ROOT / "controlled_selection_fairness_audit.csv",
        index=False,
    )
    if not fairness[
        ["same_k", "exact_total_step_parity", "same_initial_weights"]
    ].all().all():
        raise RuntimeError("CIFAR controlled-selection fairness audit failed.")

    pd.DataFrame(selection_rows).drop_duplicates().to_csv(
        EXPERIMENT_ROOT / "selection_audit.csv", index=False
    )

    pd.DataFrame([{
        "analysis_role": "TWO_SEED_DESCRIPTIVE_CROSS_DOMAIN_FEASIBILITY",
        "training_seed_count": len(TRAINING_RUN_SEEDS),
        "inferential_superiority_claimed": False,
        "reason": (
            "CIFAR-10 evaluates whether the final governance mechanism can be "
            "applied to a second data modality; two seeds are not treated as "
            "strong inferential evidence."
        ),
    }]).to_csv(
        EXPERIMENT_ROOT / "cross_domain_inference_guardrail.csv",
        index=False,
    )

    resource_cols = [
        "run", "seed", "scenario", "runtime_s",
        "energy_wh_per_w", "energy_wh", "energy_kwh", "co2_kg",
        "communication_bytes", "communication_mb", "communication_mib",
        "ram_peak_mb", "ram_delta_mb",
        "energy_cost_usd", "communication_cost_usd",
        "total_estimated_cost_usd", "optimizer_steps",
        "selected_client_count", "preprocessing_sha256",
    ]
    perf[[c for c in resource_cols if c in perf.columns]].to_csv(
        EXPERIMENT_ROOT / "resource_metrics.csv", index=False
    )

    policy = {
        "experiment_version": EXPERIMENT_VERSION,
        "scenarios": CIFAR_SCENARIOS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "fedprox_mu": FEDPROX_MU,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "HPS_weights": WEIGHTS_PSCORE_DEFAULT,
        "evidence_validity_rule": "all 28 factors present, finite, and >0",
        "dimension_min_floor": DIMENSION_MIN_FLOOR,
        "hps_reject_threshold": GOOD_CUT,
        "hps_direct_accept_threshold": HIGH_CUT,
        "critical_review_dimensions": list(CRITICAL_REVIEW_DIMS),
        "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
        "hps_review_threshold": HPS_REVIEW_THRESHOLD,
        "option_a_preprocessing": True,
        "power_of_choice_preprocessing_exception": "full TRAIN candidate pool",
        "reference_power_w": REFERENCE_POWER_W,
        "grid_carbon_intensity_kg_per_kwh": GRID_CARBON_INTENSITY_KG_PER_KWH,
        "electricity_price_usd_per_kwh": ELECTRICITY_PRICE_USD_PER_KWH,
        "communication_price_usd_per_mb": COMMUNICATION_PRICE_USD_PER_MB,
        "communication_semantics": "logical application-level payload; no universal overhead multiplier",
        "energy_semantics": "runtime-derived reference estimate; not direct Google Colab power measurement",
        "official_test_semantics": "final evaluation only",
    }
    atomic_write_json(
        policy, EXPERIMENT_ROOT / "policy_and_experiment_design.json"
    )

    print_banner("FINAL CIFAR CROSS-DOMAIN SUMMARY")
    display_cols = [
        "scenario", "n_runs",
        "accuracy_mean", "accuracy_sd",
        "f1_macro_mean", "f1_macro_sd",
        "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_sd",
        "runtime_s_mean", "communication_mb_mean",
    ]
    print(summary[[c for c in display_cols if c in summary.columns]].to_string(index=False))

    print_banner("FINAL AUDITS")
    print("No TEST leakage: PASS")
    print("Governance before model preprocessing: PASS")
    print("Scenario-specific post-selection normalization/class weights: PASS")
    print("Within-seed W0 parity: PASS")
    print("Random-K / DQ-only / PoC matched to TADP-VR K and optimizer-step budget: PASS")
    print("Two-seed descriptive cross-domain guardrail: ACTIVE")

    return EXPERIMENT_ROOT


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(
        finished_root,
        EXPERIMENT_VERSION,
    )


Google Drive checkpoint mount unavailable: mount failed
Local checkpoint root: /content/TADP_EXPERIMENT_C_FINAL_CIFAR10_OPTIONA_2SEED_4ROUND

TADP-C-FINAL-A1A2B-OPTIONA-CIFAR10-K10-2SEED-4ROUND-28F-VALIDITY-DIMFLOOR-HPSREVIEW
Training seeds: [42, 142]
FL rounds: 4
Clients: 10
Final TADP policy: 28-factor evidence validity -> dimension floor -> HPS bands -> critical-dimension HPS_review.
Cross-domain interpretation: two-seed descriptive feasibility evidence; no statistical-superiority claim.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step

NO-LEAKAGE AUDIT
 train_test_overlap  test_rows_in_any_client  train_rows_missing_from_clients  client_rows_not_in_global_train  duplicate_train_rows_across_clients  pass  official_test_untouched_before_final_evaluation  validation_excluded_from_client_partition  governance_before_model_normalization  scenario_specific_train_only_normalization  scenario_specific_train_only_class_weights  dq_gx_tadp_train_only
                  0                 

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpkezzk9x6' for ephemeral docs site


Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]


FINAL TADP POLICY — EVIDENCE VALIDITY + DIMENSION FLOOR + HPS + HPS_review
Gate 1: all 28 factors must be present, finite, and >0.
Gate 2: every dimension must be >= 2.5/5.
HPS: <3.0=Reject; [3.0,3.5)=Automated review; >=3.5=Direct accept.
HPS_review uses ('dim1', 'dim2', 'dim5', 'dim6') with weights {'dim1': 0.3125, 'dim2': 0.18749999999999997, 'dim5': 0.37499999999999994, 'dim6': 0.125}; accept if >= 3.25.
Human role: documentary-evidence verification only; factor mapping, HPS/HPS_review, and final admission are automated.

TRAIN-ONLY DATA QUALITY — 8 MACHINE-MEASURED FACTORS
client  completeness  duplication_rate  value_validity_error_rate  type_consistency  label_integrity  feature_distribution_consistency  feature_category_coverage  structural_constraint_integrity
     A           5.0               5.0                        5.0               5.0              5.0                               5.0                        5.0                              5.0
     B           5.0    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>